# Модуль 04 · Урок 3. merge, join, concat и целостность связей

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Соединение — это не простое склеивание
`merge` соединяет строки по **значению ключа**, а не по позиции. `how='inner'` возвращает только совпадающие ключи, `left` сохраняет все строки слева, `right` — справа, `outer` — объединение ключей. При `left` данные справа могут отсутствовать и стать NA. В SQL это соответствует INNER/LEFT/RIGHT/FULL OUTER JOIN.

In [ ]:
import pandas as pd
employees = pd.DataFrame({'emp_id':[1,2,3], 'dep_id':[5,6,7]})
deps = pd.DataFrame({'dep_id':[5,6],'title':['ИТ','Маркетинг']})
print(employees.merge(deps,on='dep_id',how='left',validate='many_to_one',indicator=True))

## 2. Кардинальность: главная ловушка merge
Если ключ есть 3 раза слева и 2 раза справа, соединение по нему создаст **6 строк**. Это не ошибка pandas, а декартово размножение совпадений. Используйте `validate='many_to_one'`, если справа ожидается справочник с уникальным ключом, или `one_to_one`, если уникальность требуется с обеих сторон. Число строк до/после merge обязательно сверяйте.

## 3. indicator и антисоединение
`indicator=True` добавляет `_merge`: значения `left_only`, `right_only` и `both`. Это позволяет построить отчёт о пропущенных ссылках. Например, после left merge можно выбрать строки `_merge == 'left_only'` и найти заказы с несуществующими клиентами. Проверка ссылочной целостности не должна незаметно удалять такие строки через inner join.

In [ ]:
left = pd.DataFrame({'key':['a','b','c']})
right = pd.DataFrame({'key':['b','c','d']})
print(left.merge(right,on='key',how='outer',indicator=True))

## 4. concat, join и merge_asof
`pd.concat([df1,df2],ignore_index=True)` складывает таблицы **по вертикали**; при разных колонках появляются NA. `DataFrame.join` часто используется для соединения по индексам. `merge_asof` соединяет ближайшие точки по отсортированным временным ключам, полезно для котировок и событий, но не является обычным равенством ключей.

**Проверьте себя:** как проверить, что справочник справа не размножил заказы? В каком случае outer join создаёт NA? Зачем `_merge`?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m4_11 · attach clients

Обогатите заказы колонками справочника клиентов через LEFT merge по client_id. Предполагается, что ключ справочника уникален.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `reference` — `pd.DataFrame`, колонки: `client_id`, `segment`, `region`.


**Выполнение:** используйте готовые переменные `df`, `reference`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_11: используются переменные df, reference

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

reference = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})

print("df — демонстрационные данные:")

display(df)

print("reference — демонстрационные данные:")

display(reference)


# Снимок правильного результата для автопроверки.
_expected_m4_11 = expected_for("m4_11", df=df, reference=reference)


In [ ]:
# m4_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, reference
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 12 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `segment`, `region`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment segment        region
0  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...     vip  Центр (демо)
1  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка    base  Север (демо)
2  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно     vip  Центр (демо)
3  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок     vip     Юг (демо)
4  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                     base  Север (демо)
5  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None     NaN           NaN
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_11", result, _expected_m4_11)

### m4_12 · missing clients

Получите DataFrame только с колонками `order_id`,`client_id` для заказов, клиент которых отсутствует в справочнике. Индекс результата сбросьте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `reference` — `pd.DataFrame`, колонки: `client_id`, `segment`, `region`.


**Выполнение:** используйте готовые переменные `df`, `reference`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_12: используются переменные df, reference

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

reference = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})

print("df — демонстрационные данные:")

display(df)

print("reference — демонстрационные данные:")

display(reference)


# Снимок правильного результата для автопроверки.
_expected_m4_12 = expected_for("m4_12", df=df, reference=reference)


In [ ]:
# m4_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, reference
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 1 строка, 2 столбца; колонки: `order_id`, `client_id`.

```text
  order_id  client_id
0  demo_o6          4
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_12", result, _expected_m4_12)

### m4_13 · combine batches

Сложите два пакета заказов вертикально, сбросьте индекс и сохраните все колонки.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `a` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `b` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `a`, `b`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_13: используются переменные a, b

a = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3'],
    'client_id': [1, 2, 1],
    'city': ['Москва', 'Тверь', 'Москва'],
    'category': ['чай', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0],
    'qty': [2, 1, 4],
    'discount': [0, 10, 5],
    'status': ['paid', 'new', 'paid'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно'],
}, index=['z', 'b', 'a'])

b = pd.DataFrame({
    'order_id': ['demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [3, 2, 4],
    'city': ['Казань', 'Тверь', 'Казань'],
    'category': ['книги', 'кофе', 'чай'],
    'price': [317.0, 57.0, 147.0],
    'qty': [1, 3, 0],
    'discount': [0, 20, 0],
    'status': ['paid', 'cancelled', 'new'],
    'date': ['2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['подарок', '', None],
}, index=['d', 'x', 'q'])

print("a — демонстрационные данные:")

display(a)

print("b — демонстрационные данные:")

display(b)


# Снимок правильного результата для автопроверки.
_expected_m4_13 = expected_for("m4_13", a=a, b=b)


In [ ]:
# m4_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: a, b
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
0  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
1  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
2  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
3  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
4  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
5  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_13", result, _expected_m4_13)

### m4_14 · outer status

Сделайте полный OUTER merge таблиц по `client_id`, оставьте индикатор `_merge` с исходными категориями.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `left` — `pd.DataFrame`, колонки: `client_id`, `order_id`.
- `right` — `pd.DataFrame`, колонки: `client_id`, `segment`, `region`.


**Выполнение:** используйте готовые переменные `left`, `right`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_14: используются переменные left, right

left = pd.DataFrame({
    'client_id': [1, 2, 1, 3, 2, 4],
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

right = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})

print("left — демонстрационные данные:")

display(left)

print("right — демонстрационные данные:")

display(right)


# Снимок правильного результата для автопроверки.
_expected_m4_14 = expected_for("m4_14", left=left, right=right)


In [ ]:
# m4_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: left, right
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 7 строк, 5 столбцов; колонки: `client_id`, `order_id`, `segment`, `region`, `_merge`.

```text
   client_id order_id segment        region     _merge
0          1  demo_o1     vip  Центр (демо)       both
1          1  demo_o3     vip  Центр (демо)       both
2          2  demo_o2    base  Север (демо)       both
3          2  demo_o5    base  Север (демо)       both
4          3  demo_o4     vip     Юг (демо)       both
5          4  demo_o6     NaN           NaN  left_only
... (показаны только первые строки)
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_14", result, _expected_m4_14)

### m4_15 · attach on index

Возьмите список заказов и справочник клиентов, соедините справочник по client_id, используя join по индексу справочника. В результате сохраняются исходные индексы заказов.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `reference` — `pd.DataFrame`, колонки: `client_id`, `segment`, `region`.


**Выполнение:** используйте готовые переменные `df`, `reference`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_15: используются переменные df, reference

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

reference = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})

print("df — демонстрационные данные:")

display(df)

print("reference — демонстрационные данные:")

display(reference)


# Снимок правильного результата для автопроверки.
_expected_m4_15 = expected_for("m4_15", df=df, reference=reference)


In [ ]:
# m4_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, reference
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 12 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `segment`, `region`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment segment        region
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...     vip  Центр (демо)
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка    base  Север (демо)
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно     vip  Центр (демо)
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок     vip     Юг (демо)
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                     base  Север (демо)
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None     NaN           NaN
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_15", result, _expected_m4_15)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()